# Corrección de valores anómalos en curvas de luz

In [1]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq

In [2]:
# --------------------------------------------------------------
# Rutas de entrada
# --------------------------------------------------------------

# Parquet plegado que ya incorpora el período fundamental
# para las 22 estrellas aRRd.
ruta_curvas_plegadas_entrada = Path(
    "../data/interim/curvas_plegadas_corregidas.parquet"
)

# Catálogo asociado al Parquet de entrada.
ruta_catalogo_entrada = Path(
    "../data/interim/catalogo_maestro_corregido.csv"
)


# --------------------------------------------------------------
# Rutas de salida
# --------------------------------------------------------------

# Parquet definitivo después de retirar las 122 mediciones
# fotométricas incompatibles.
ruta_curvas_plegadas_final = Path(
    "../data/interim/curvas_plegadas_final.parquet"
)

# Catálogo definitivo con los conteos de mediciones actualizados.
ruta_catalogo_final = Path(
    "../data/interim/catalogo_maestro_final.csv"
)

# Registro independiente de las curvas y mediciones modificadas.
ruta_auditoria_fotometrica = Path(
    "../data/interim/auditoria_fotometria_anomala.csv"
)


# --------------------------------------------------------------
# Parámetros de la corrección
# --------------------------------------------------------------

# Curvas en las que se detectaron las 122 mediciones incompatibles
# con su escala fotométrica habitual.
ids_fotometria_anomala = {
    "OGLEIII_OGLE-LMC-RRLYR-14167",
    "OGLEIII_OGLE-LMC-RRLYR-17594",
}

# Umbral utilizado exclusivamente en las dos curvas identificadas.
# La auditoría previa mostró que ninguna otra curva contiene
# magnitudes inferiores a este valor.
umbral_magnitud_valida = 10.0


# --------------------------------------------------------------
# Comprobación de archivos de entrada
# --------------------------------------------------------------

print(
    "Parquet de entrada encontrado:",
    ruta_curvas_plegadas_entrada.exists(),
)

print(
    "Catálogo de entrada encontrado:",
    ruta_catalogo_entrada.exists(),
)

print(
    "Ruta del Parquet final:",
    ruta_curvas_plegadas_final.resolve(),
)

print(
    "Ruta del catálogo final:",
    ruta_catalogo_final.resolve(),
)

print(
    "Ruta de la auditoría:",
    ruta_auditoria_fotometrica.resolve(),
)

Parquet de entrada encontrado: True
Catálogo de entrada encontrado: True
Ruta del Parquet final: C:\Users\adanm\Code\tesis\data\interim\curvas_plegadas_final.parquet
Ruta del catálogo final: C:\Users\adanm\Code\tesis\data\interim\catalogo_maestro_final.csv
Ruta de la auditoría: C:\Users\adanm\Code\tesis\data\interim\auditoria_fotometria_anomala.csv


In [3]:
# --------------------------------------------------------------
# Carga de los archivos de entrada
# --------------------------------------------------------------

catalogo_entrada = pd.read_csv(
    ruta_catalogo_entrada
)

archivo_origen_plegado = pq.ParquetFile(
    ruta_curvas_plegadas_entrada
)

esquema_plegado = (
    archivo_origen_plegado
    .schema_arrow
)

print(
    "Filas del catálogo:",
    f"{len(catalogo_entrada):,}",
)

print(
    "Filas del Parquet:",
    f"{archivo_origen_plegado.metadata.num_rows:,}",
)

print(
    "Grupos internos:",
    archivo_origen_plegado.num_row_groups,
)


# --------------------------------------------------------------
# Inicialización de la auditoría
# --------------------------------------------------------------

registros_correccion_fotometrica = []

filas_procesadas = 0
mediciones_originales_totales = 0
mediciones_finales_totales = 0
mediciones_retiradas_totales = 0

conteos_finales_por_id = {}


# --------------------------------------------------------------
# Creación del Parquet plegado final
# --------------------------------------------------------------

with pq.ParquetWriter(
    ruta_curvas_plegadas_final,
    esquema_plegado,
    compression="zstd",
) as escritor_final:

    for numero_grupo in range(
        archivo_origen_plegado.num_row_groups
    ):

        tabla_grupo = (
            archivo_origen_plegado
            .read_row_group(
                numero_grupo
            )
        )

        filas_grupo = tabla_grupo.to_pylist()

        for fila in filas_grupo:

            observacion_id = str(
                fila["observacion_id"]
            )

            fase = np.asarray(
                fila["fase"],
                dtype=np.float64,
            )

            magnitud = np.asarray(
                fila["magnitud"],
                dtype=np.float32,
            )

            error = np.asarray(
                fila["error"],
                dtype=np.float32,
            )

            n_original = len(
                magnitud
            )

            mediciones_originales_totales += (
                n_original
            )

            # El filtro se aplica exclusivamente a las dos curvas
            # previamente identificadas mediante la auditoría.
            if (
                observacion_id
                in ids_fotometria_anomala
            ):

                mascara_valida = (
                    magnitud
                    >= umbral_magnitud_valida
                )

                n_retiradas = int(
                    (
                        ~mascara_valida
                    ).sum()
                )

                magnitudes_retiradas = (
                    magnitud[
                        ~mascara_valida
                    ]
                )

                errores_retirados = (
                    error[
                        ~mascara_valida
                    ]
                )

                fase = fase[
                    mascara_valida
                ]

                magnitud = magnitud[
                    mascara_valida
                ]

                error = error[
                    mascara_valida
                ]

                n_final = len(
                    magnitud
                )

                if n_retiradas == 0:
                    raise ValueError(
                        f"No se encontraron mediciones "
                        f"anómalas en {observacion_id}."
                    )

                if n_final < 300:
                    raise ValueError(
                        f"{observacion_id} quedó con "
                        f"{n_final} mediciones, por debajo "
                        "del umbral mínimo de 300."
                    )

                registros_correccion_fotometrica.append({
                    "observacion_id":
                        observacion_id,
                    "n_mediciones_originales":
                        n_original,
                    "n_mediciones_retiradas":
                        n_retiradas,
                    "porcentaje_retirado":
                        (
                            100.0
                            * n_retiradas
                            / n_original
                        ),
                    "n_mediciones_finales":
                        n_final,
                    "magnitud_retirada_minima":
                        float(
                            magnitudes_retiradas.min()
                        ),
                    "magnitud_retirada_maxima":
                        float(
                            magnitudes_retiradas.max()
                        ),
                    "error_retirado_minimo":
                        float(
                            errores_retirados.min()
                        ),
                    "error_retirado_maximo":
                        float(
                            errores_retirados.max()
                        ),
                    "umbral_aplicado":
                        umbral_magnitud_valida,
                })

            else:

                n_retiradas = 0
                n_final = n_original

            # Las fases, magnitudes y errores deben permanecer alineados.
            if not (
                len(fase)
                == len(magnitud)
                == len(error)
            ):
                raise ValueError(
                    f"Desalineación detectada en "
                    f"{observacion_id}."
                )

            # La eliminación conserva el orden original de las fases.
            fila["fase"] = (
                fase
                .astype(
                    np.float64
                )
                .tolist()
            )

            fila["magnitud"] = (
                magnitud
                .astype(
                    np.float32
                )
                .tolist()
            )

            fila["error"] = (
                error
                .astype(
                    np.float32
                )
                .tolist()
            )

            fila["n_obs_folding"] = int(
                n_final
            )

            conteos_finales_por_id[
                observacion_id
            ] = n_final

            mediciones_retiradas_totales += (
                n_retiradas
            )

            mediciones_finales_totales += (
                n_final
            )

            filas_procesadas += 1

        tabla_grupo_final = pa.Table.from_pylist(
            filas_grupo,
            schema=esquema_plegado,
        )

        escritor_final.write_table(
            tabla_grupo_final
        )

        if (
            (numero_grupo + 1) % 20 == 0
            or numero_grupo + 1
            == archivo_origen_plegado.num_row_groups
        ):
            print(
                f"{numero_grupo + 1} de "
                f"{archivo_origen_plegado.num_row_groups} "
                "grupos procesados"
            )


# --------------------------------------------------------------
# Actualización del catálogo maestro
# --------------------------------------------------------------

catalogo_final = catalogo_entrada.copy()

# Actualizamos todas las columnas existentes que representan
# la cantidad de mediciones válidas o plegadas.
columnas_conteo_catalogo = [
    columna
    for columna in [
        "n_obs_validas",
        "n_obs_folding",
    ]
    if columna in catalogo_final.columns
]

mascara_curvas_afectadas = (
    catalogo_final[
        "observacion_id"
    ].isin(
        ids_fotometria_anomala
    )
)

for columna in columnas_conteo_catalogo:

    catalogo_final.loc[
        mascara_curvas_afectadas,
        columna,
    ] = (
        catalogo_final.loc[
            mascara_curvas_afectadas,
            "observacion_id",
        ]
        .map(
            conteos_finales_por_id
        )
        .astype(int)
    )

catalogo_final.to_csv(
    ruta_catalogo_final,
    index=False,
)


# --------------------------------------------------------------
# Almacenamiento de la auditoría
# --------------------------------------------------------------

auditoria_correccion_fotometrica = pd.DataFrame(
    registros_correccion_fotometrica
)

auditoria_correccion_fotometrica.to_csv(
    ruta_auditoria_fotometrica,
    index=False,
)


# --------------------------------------------------------------
# Validaciones inmediatas
# --------------------------------------------------------------

archivo_final_plegado = pq.ParquetFile(
    ruta_curvas_plegadas_final
)

mediciones_finales_esperadas = (
    mediciones_originales_totales
    - mediciones_retiradas_totales
)

if filas_procesadas != 104_008:
    raise ValueError(
        "La cantidad final de curvas no es 104.008."
    )

if mediciones_retiradas_totales != 122:
    raise ValueError(
        "No se retiraron exactamente las 122 "
        "mediciones esperadas."
    )

if (
    mediciones_finales_totales
    != mediciones_finales_esperadas
):
    raise ValueError(
        "El total final de mediciones no coincide."
    )

if (
    mediciones_finales_totales
    != 132_232_695
):
    raise ValueError(
        "El archivo final no contiene las "
        "132.232.695 mediciones esperadas."
    )

if (
    archivo_final_plegado.metadata.num_rows
    != 104_008
):
    raise ValueError(
        "El Parquet final no contiene 104.008 curvas."
    )

if len(
    auditoria_correccion_fotometrica
) != 2:
    raise ValueError(
        "La auditoría no contiene exactamente "
        "las dos curvas modificadas."
    )


# --------------------------------------------------------------
# Resumen
# --------------------------------------------------------------

print(
    "Parquet final:",
    ruta_curvas_plegadas_final.resolve(),
)

print(
    "Catálogo final:",
    ruta_catalogo_final.resolve(),
)

print(
    "Auditoría:",
    ruta_auditoria_fotometrica.resolve(),
)

print(
    "Curvas conservadas:",
    f"{filas_procesadas:,}",
)

print(
    "Mediciones originales:",
    f"{mediciones_originales_totales:,}",
)

print(
    "Mediciones retiradas:",
    f"{mediciones_retiradas_totales:,}",
)

print(
    "Mediciones finales:",
    f"{mediciones_finales_totales:,}",
)

print(
    "Columnas actualizadas en el catálogo:",
    columnas_conteo_catalogo,
)

display(
    auditoria_correccion_fotometrica
)

C:\Users\adanm\AppData\Local\Temp\ipykernel_11780\1835804266.py:5: DtypeWarning: Columns (0: transicion) have mixed types. Specify dtype option on import or set low_memory=False.
  catalogo_entrada = pd.read_csv(


Filas del catálogo: 104,008
Filas del Parquet: 104,008
Grupos internos: 105
20 de 105 grupos procesados
40 de 105 grupos procesados
60 de 105 grupos procesados
80 de 105 grupos procesados
100 de 105 grupos procesados
105 de 105 grupos procesados
Parquet final: C:\Users\adanm\Code\tesis\data\interim\curvas_plegadas_final.parquet
Catálogo final: C:\Users\adanm\Code\tesis\data\interim\catalogo_maestro_final.csv
Auditoría: C:\Users\adanm\Code\tesis\data\interim\auditoria_fotometria_anomala.csv
Curvas conservadas: 104,008
Mediciones originales: 132,232,817
Mediciones retiradas: 122
Mediciones finales: 132,232,695
Columnas actualizadas en el catálogo: ['n_obs_validas', 'n_obs_folding']


,observacion_id,n_mediciones_originales,n_mediciones_retiradas,porcentaje_retirado,n_mediciones_finales,magnitud_retirada_minima,magnitud_retirada_maxima,error_retirado_minimo,error_retirado_maximo,umbral_aplicado
0,OGLEIII_OGLE-LMC-RRLYR-14167,891,120,13.468013,771,0.013,0.013,0.005,0.005,10.0
1,OGLEIII_OGLE-LMC-RRLYR-17594,1029,2,0.194363,1027,-0.002,-0.002,2.538,4.546,10.0


In [4]:
# --------------------------------------------------------------
# Carga de los archivos finales
# --------------------------------------------------------------

archivo_final_plegado = pq.ParquetFile(
    ruta_curvas_plegadas_final
)

catalogo_final_validacion = pd.read_csv(
    ruta_catalogo_final
)

archivo_original_plegado = pq.ParquetFile(
    ruta_curvas_plegadas_entrada
)

if (
    archivo_original_plegado.num_row_groups
    != archivo_final_plegado.num_row_groups
):
    raise ValueError(
        "Los archivos original y final no tienen "
        "la misma cantidad de grupos internos."
    )


# --------------------------------------------------------------
# Comparación completa del Parquet original y el final
# --------------------------------------------------------------

filas_originales = 0
filas_finales = 0

mediciones_originales = 0
mediciones_finales = 0

identificadores_desalineados = 0
curvas_no_objetivo_modificadas = 0
curvas_objetivo_validadas = 0
curvas_objetivo_incorrectas = 0

fases_fuera_intervalo = 0
errores_no_positivos = 0
magnitudes_inferiores_umbral = 0
valores_no_finitos = 0

conteos_parquet_final = {}
magnitud_minima_final = np.inf
magnitud_maxima_final = -np.inf

for numero_grupo in range(
    archivo_original_plegado.num_row_groups
):

    tabla_original = (
        archivo_original_plegado
        .read_row_group(
            numero_grupo
        )
        .to_pydict()
    )

    tabla_final = (
        archivo_final_plegado
        .read_row_group(
            numero_grupo
        )
        .to_pydict()
    )

    cantidad_filas_original = len(
        tabla_original["observacion_id"]
    )

    cantidad_filas_final = len(
        tabla_final["observacion_id"]
    )

    filas_originales += cantidad_filas_original
    filas_finales += cantidad_filas_final

    if (
        cantidad_filas_original
        != cantidad_filas_final
    ):
        raise ValueError(
            f"El grupo {numero_grupo} cambió "
            "su cantidad de curvas."
        )

    for indice_fila in range(
        cantidad_filas_original
    ):

        id_original = str(
            tabla_original[
                "observacion_id"
            ][indice_fila]
        )

        id_final = str(
            tabla_final[
                "observacion_id"
            ][indice_fila]
        )

        if id_original != id_final:
            identificadores_desalineados += 1
            continue

        fase_original = np.asarray(
            tabla_original[
                "fase"
            ][indice_fila],
            dtype=np.float64,
        )

        magnitud_original = np.asarray(
            tabla_original[
                "magnitud"
            ][indice_fila],
            dtype=np.float32,
        )

        error_original = np.asarray(
            tabla_original[
                "error"
            ][indice_fila],
            dtype=np.float32,
        )

        fase_final = np.asarray(
            tabla_final[
                "fase"
            ][indice_fila],
            dtype=np.float64,
        )

        magnitud_final = np.asarray(
            tabla_final[
                "magnitud"
            ][indice_fila],
            dtype=np.float32,
        )

        error_final = np.asarray(
            tabla_final[
                "error"
            ][indice_fila],
            dtype=np.float32,
        )

        n_original = len(
            fase_original
        )

        n_final = len(
            fase_final
        )

        mediciones_originales += n_original
        mediciones_finales += n_final

        conteos_parquet_final[
            id_final
        ] = n_final

        # ------------------------------------------------------
        # Validación de las curvas que no debían cambiar
        # ------------------------------------------------------
        if (
            id_final
            not in ids_fotometria_anomala
        ):

            contenido_igual = (
                np.array_equal(
                    fase_original,
                    fase_final,
                )
                and np.array_equal(
                    magnitud_original,
                    magnitud_final,
                )
                and np.array_equal(
                    error_original,
                    error_final,
                )
                and n_original == n_final
            )

            if not contenido_igual:
                curvas_no_objetivo_modificadas += 1

        # ------------------------------------------------------
        # Validación de las dos curvas corregidas
        # ------------------------------------------------------
        else:

            mascara_esperada = (
                magnitud_original
                >= umbral_magnitud_valida
            )

            fase_esperada = (
                fase_original[
                    mascara_esperada
                ]
            )

            magnitud_esperada = (
                magnitud_original[
                    mascara_esperada
                ]
            )

            error_esperado = (
                error_original[
                    mascara_esperada
                ]
            )

            correccion_correcta = (
                np.array_equal(
                    fase_esperada,
                    fase_final,
                )
                and np.array_equal(
                    magnitud_esperada,
                    magnitud_final,
                )
                and np.array_equal(
                    error_esperado,
                    error_final,
                )
                and n_final
                == int(
                    mascara_esperada.sum()
                )
            )

            if correccion_correcta:
                curvas_objetivo_validadas += 1
            else:
                curvas_objetivo_incorrectas += 1

        # ------------------------------------------------------
        # Validación general de los valores finales
        # ------------------------------------------------------
        fases_fuera_intervalo += int(
            (
                (fase_final < 0)
                | (fase_final >= 1)
            ).sum()
        )

        errores_no_positivos += int(
            (
                error_final <= 0
            ).sum()
        )

        magnitudes_inferiores_umbral += int(
            (
                magnitud_final
                < umbral_magnitud_valida
            ).sum()
        )

        valores_no_finitos += int(
            (
                ~np.isfinite(
                    fase_final
                )
            ).sum()
            + (
                ~np.isfinite(
                    magnitud_final
                )
            ).sum()
            + (
                ~np.isfinite(
                    error_final
                )
            ).sum()
        )

        magnitud_minima_final = min(
            magnitud_minima_final,
            float(
                magnitud_final.min()
            ),
        )

        magnitud_maxima_final = max(
            magnitud_maxima_final,
            float(
                magnitud_final.max()
            ),
        )

    if (
        (numero_grupo + 1) % 20 == 0
        or numero_grupo + 1
        == archivo_original_plegado.num_row_groups
    ):
        print(
            f"{numero_grupo + 1} de "
            f"{archivo_original_plegado.num_row_groups} "
            "grupos validados"
        )


# --------------------------------------------------------------
# Validación del catálogo contra el Parquet final
# --------------------------------------------------------------

conteos_catalogo = (
    catalogo_final_validacion[
        [
            "observacion_id",
            "n_obs_folding",
        ]
    ]
    .copy()
)

conteos_catalogo[
    "n_obs_parquet"
] = (
    conteos_catalogo[
        "observacion_id"
    ]
    .map(
        conteos_parquet_final
    )
)

diferencias_n_obs_folding = int(
    (
        conteos_catalogo[
            "n_obs_folding"
        ]
        != conteos_catalogo[
            "n_obs_parquet"
        ]
    ).sum()
)

if (
    "n_obs_validas"
    in catalogo_final_validacion.columns
):

    diferencias_n_obs_validas = int(
        (
            catalogo_final_validacion[
                "n_obs_validas"
            ]
            != catalogo_final_validacion[
                "observacion_id"
            ].map(
                conteos_parquet_final
            )
        ).sum()
    )

else:
    diferencias_n_obs_validas = None


# --------------------------------------------------------------
# Resumen de la validación
# --------------------------------------------------------------

resumen_validacion_final = pd.DataFrame({
    "metrica": [
        "Filas originales",
        "Filas finales",
        "Mediciones originales",
        "Mediciones finales",
        "Diferencia de mediciones",
        "Identificadores desalineados",
        "Curvas no objetivo modificadas",
        "Curvas objetivo validadas",
        "Curvas objetivo incorrectas",
        "Fases fuera de [0,1)",
        "Errores no positivos",
        "Magnitudes inferiores a 10",
        "Valores no finitos",
        "Magnitud mínima final",
        "Magnitud máxima final",
        "Diferencias en n_obs_folding",
        "Diferencias en n_obs_validas",
        "Identificadores únicos en el catálogo",
        "Identificadores únicos en el Parquet",
    ],
    "valor": [
        filas_originales,
        filas_finales,
        mediciones_originales,
        mediciones_finales,
        (
            mediciones_originales
            - mediciones_finales
        ),
        identificadores_desalineados,
        curvas_no_objetivo_modificadas,
        curvas_objetivo_validadas,
        curvas_objetivo_incorrectas,
        fases_fuera_intervalo,
        errores_no_positivos,
        magnitudes_inferiores_umbral,
        valores_no_finitos,
        magnitud_minima_final,
        magnitud_maxima_final,
        diferencias_n_obs_folding,
        diferencias_n_obs_validas,
        catalogo_final_validacion[
            "observacion_id"
        ].nunique(),
        len(
            conteos_parquet_final
        ),
    ],
})

display(
    resumen_validacion_final
)

C:\Users\adanm\AppData\Local\Temp\ipykernel_11780\3330306924.py:9: DtypeWarning: Columns (0: transicion) have mixed types. Specify dtype option on import or set low_memory=False.
  catalogo_final_validacion = pd.read_csv(


20 de 105 grupos validados
40 de 105 grupos validados
60 de 105 grupos validados
80 de 105 grupos validados
100 de 105 grupos validados
105 de 105 grupos validados


,metrica,valor
0,Filas originales,1.040080e+05
1,Filas finales,1.040080e+05
2,Mediciones originales,1.322328e+08
3,Mediciones finales,1.322327e+08
4,Diferencia de mediciones,1.220000e+02
5,Identificadores desalineados,0.000000e+00
6,Curvas no objetivo modificadas,0.000000e+00
7,Curvas objetivo validadas,2.000000e+00
8,Curvas objetivo incorrectas,0.000000e+00
9,"Fases fuera de [0,1)",0.000000e+00
